**FITS → CSV + flags**

In [ ]:
from glob import glob
import os
import pandas as pd
from astropy.table import Table

# Convert hunt.fit/hunt.fits to CSV 
paths = glob("**/hunt.fit", recursive=True) + glob("**/hunt.fits", recursive=True)
assert paths, "No hunt.fit or hunt.fits found."
inpath = sorted(paths, key=len)[0]

table = Table.read(inpath)
hunt_df = table.to_pandas()

base = os.path.join(os.path.dirname(inpath), "hunt")
hunt_csv = base + ".csv"
i = 1
while os.path.exists(hunt_csv):
    hunt_csv = f"{base}_{i}.csv"
    i += 1

hunt_df.to_csv(hunt_csv, index=False)
print(f"Converted: {inpath} -> {hunt_csv}")

# Add Gaia/TESScrossmatch + Within1kpc flags 
filtered_path = "filtered_hunt.csv"
assert os.path.exists(filtered_path), "filtered_hunt.csv not found in current directory."

EXPECTED_ROWS = 1291929
assert len(hunt_df) == EXPECTED_ROWS, f"{os.path.basename(hunt_csv)} has {len(hunt_df)} rows, expected {EXPECTED_ROWS}."

# Preserve Gaia IDs as strings
assert "GaiaDR3" in hunt_df.columns, "GaiaDR3 column missing from FITS table."
hunt_df["GaiaDR3"] = hunt_df["GaiaDR3"].astype(str)

filtered = pd.read_csv(filtered_path, dtype={"GaiaDR3": str})
filtered_ids = filtered[["GaiaDR3"]].dropna().drop_duplicates()

# Left-merge membership flag only (preserve all of my rows)
merged = hunt_df.merge(filtered_ids.assign(_in_filtered=1), on="GaiaDR3", how="left")

merged["Gaia/TESScrossmatch"] = "y"  # every hunt row marked 'y'
merged["Within1kpc"] = merged["_in_filtered"].fillna(0).astype(int).map({1: "y", 0: "n"})
merged = merged.drop(columns=["_in_filtered"])

# Save without overwriting
out_base = os.path.join(os.path.dirname(hunt_csv), "hunt_with_flags.csv")
out_path = out_base
i = 1
while os.path.exists(out_path):
    out_path = out_base.replace(".csv", f"_{i}.csv")
    i += 1

merged.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows: {len(merged)} (expected {EXPECTED_ROWS})")
print("Counts Within1kpc -> y:", (merged["Within1kpc"] == "y").sum(),
      " n:", (merged["Within1kpc"] == "n").sum())


**Gaia mapping + Mag<13 + Giants**

In [ ]:
import os, pandas as pd
from glob import glob

EXPECTED_ROWS = 1291929

# Locate latest hunt_with_flags*.csv from Cell 1 
cands = glob("**/hunt_with_flags*.csv", recursive=True)
assert cands, "No hunt_with_flags*.csv found. Run Cell 1 first."
hunt_path = max(cands, key=os.path.getmtime)

mapping_path = "gaia_fit_mapping_hunt.csv"
cls_path = "classified_stars.csv"
assert os.path.exists(mapping_path), "gaia_fit_mapping_hunt.csv not found."
assert os.path.exists(cls_path), "classified_stars.csv not found."

# Load (keep Gaia IDs as strings) 
hunt = pd.read_csv(hunt_path, dtype={"GaiaDR3": str})
mapping = pd.read_csv(mapping_path, dtype={"GaiaDR3": str})
cls = pd.read_csv(cls_path, dtype={"GaiaDR3": str})

assert len(hunt) == EXPECTED_ROWS, f"{os.path.basename(hunt_path)} has {len(hunt)} rows; expected {EXPECTED_ROWS}."
assert "Gmag" in hunt.columns, "'Gmag' column not found in hunt file."

# Mag<13/TESS flag 
hunt["Gmag"] = pd.to_numeric(hunt["Gmag"], errors="coerce")
hunt["Mag<13/TESS"] = hunt["Gmag"].lt(13).map({True: "y", False: "n"})

# De-duplicate Gaia mapping to prevent row explosion 
before_map = len(mapping)
mapping = mapping.dropna(subset=["GaiaDR3"]).drop_duplicates(subset=["GaiaDR3"], keep="first")
dedup_map_removed = before_map - len(mapping)

# Merge mapping
merged = hunt.merge(mapping, on="GaiaDR3", how="left", suffixes=("", "_map"))
assert len(merged) == EXPECTED_ROWS, f"Mapping merge produced {len(merged)} rows; expected {EXPECTED_ROWS}."

# De-duplicate classification file 
before_cls = len(cls)
cls = cls.dropna(subset=["GaiaDR3"]).drop_duplicates(subset=["GaiaDR3"], keep="first")
dedup_cls_removed = before_cls - len(cls)
assert "Classification" in cls.columns, "'Classification' column missing in classified_stars.csv."

# Merge classification
merged = merged.merge(cls[["GaiaDR3", "Classification"]], on="GaiaDR3", how="left")
assert len(merged) == EXPECTED_ROWS, f"Classification merge produced {len(merged)} rows; expected {EXPECTED_ROWS}."

# Giants flag 
is_prg = merged["Classification"].astype(str).str.strip().str.casefold().eq("potential red giant")
merged["Giants"] = is_prg.map({True: "y", False: "n"})

# Save without overwriting 
base = os.path.join(os.path.dirname(hunt_path), "hunt_with_flags_mag_map_classified.csv")
out = base
i = 1
while os.path.exists(out):
    out = base.replace(".csv", f"_{i}.csv")
    i += 1

merged.to_csv(out, index=False)

print(f"Saved: {out}")
print(f"Rows: {len(merged)} (expected {EXPECTED_ROWS})")
print(f"Mapping duplicates removed: {dedup_map_removed}")
print(f"Classified duplicates removed: {dedup_cls_removed}")
print("Giants counts -> y:", (merged["Giants"] == "y").sum(),
      " n:", (merged["Giants"] == "n").sum())


**Within1kpc subset + flag sort**

In [ ]:
import os, pandas as pd
from glob import glob

# Find latest merged file from Cell 2 (prefer classified) 
cands = glob("**/hunt_with_flags*_classified*.csv", recursive=True) or \
        glob("**/hunt_with_flags*.csv", recursive=True)
assert cands, "No merged hunt_with_flags*.csv found. Run Cells 1–2 first."
inpath = max(cands, key=os.path.getmtime)

df = pd.read_csv(inpath, dtype={"GaiaDR3": str})
assert "Within1kpc" in df.columns, "'Within1kpc' column missing."

# Keep only Within1kpc == 'y' 
df1 = df[df["Within1kpc"].astype(str).str.strip().str.lower().eq("y")].copy()

# Drop specified columns (ignore if missing) 
cols_to_drop = ["Gaia/TESScrossmatch", "inrt", "Prob", "ID", "_RA_icrs", "_DE_icrs", "recno"]
drop_actual = [c for c in cols_to_drop if c in df1.columns]
df1.drop(columns=drop_actual, inplace=True, errors="ignore")

# Save clean Within1kpc subset 
base = os.path.join(os.path.dirname(inpath), "hunt_within1kpc_clean.csv")
out1 = base
i = 1
while os.path.exists(out1):
    out1 = base.replace(".csv", f"_{i}.csv")
    i += 1
df1.to_csv(out1, index=False)

print(f"Input:  {inpath}")
print(f"Dropped columns: {drop_actual}")
print(f"Saved clean Within1kpc subset: {out1}")
print(f"Rows kept (Within1kpc='y'): {len(df1)}")

# Sort by number of 'y' flags across key columns 
flags = ["Within1kpc", "Mag<13/TESS", "Giants"]
missing = [c for c in flags if c not in df1.columns]
assert not missing, f"Missing columns when sorting by flags: {missing}"

ycount = df1[flags].apply(
    lambda s: (s.astype(str).str.strip().str.lower() == "y").sum(),
    axis=1,
)
df_sorted = df1.assign(_Ycount=ycount).sort_values("_Ycount", ascending=False).drop(columns="_Ycount")

# Save sorted version 
base2 = os.path.join(os.path.dirname(out1), "hunt_sorted_by_flags.csv")
out2 = base2
i = 1
while os.path.exists(out2):
    out2 = base2.replace(".csv", f"_{i}.csv")
    i += 1
df_sorted.to_csv(out2, index=False)

counts = ycount.value_counts().reindex([3, 2, 1, 0], fill_value=0)
print(f"Saved sorted file: {out2}")
print("Rows by number of 'y' flags (3,2,1,0):", tuple(int(counts[k]) for k in [3, 2, 1, 0]))


**Clean rankings + merge into HUNT CSV**

In [ ]:
import pandas as pd, re, os, numpy as np
from glob import glob

# Optional: clean raw Starrank.csv if present 
rank_raw_path = "Starrank.csv"
if os.path.exists(rank_raw_path):
    df_rank = pd.read_csv(rank_raw_path, engine="python")

    # 1) Drop Excel "Unnamed:" artifact columns
    keep = [c for c in df_rank.columns if not str(c).lower().startswith("unnamed:")]
    df_rank = df_rank[keep].copy()

    # 2) Normalize column names (strip, collapse spaces, fix NBSP)
    def norm(c):
        c = str(c).replace("\u00A0", " ")
        c = re.sub(r"\s+"," ", c).strip()
        return c
    df_rank.columns = [norm(c) for c in df_rank.columns]

    # 3) Standardize y/n-like columns
    for c in df_rank.columns:
        if df_rank[c].dtype == object:
            s = df_rank[c].astype(str).str.replace("\u00A0"," ", regex=False).str.strip().str.lower()
            if s.isin(["y","n","yes","no","true","false"]).mean() > 0.5:
                df_rank[c] = s.map({
                    "y": "y", "yes": "y", "true": "y",
                    "n": "n", "no": "n", "false": "n"
                }).fillna(df_rank[c])

    # 4) Trim key columns
    for key in ["GaiaDR3","TICID"]:
        if key in df_rank.columns:
            df_rank[key] = df_rank[key].astype(str).str.strip()

    # 5) De-duplicate by TICID if it exists
    if "TICID" in df_rank.columns:
        df_rank = df_rank.sort_values(list(df_rank.columns))
        df_rank = (df_rank.groupby("TICID", as_index=False)
                          .agg(lambda s: s.dropna().iloc[0] if s.dropna().size else pd.NA))

    out_rank = os.path.join(os.path.dirname(rank_raw_path), "Star_Rankings_clean.csv")
    i = 1
    while os.path.exists(out_rank):
        out_rank = os.path.join(os.path.dirname(rank_raw_path), f"Star_Rankings_clean_{i}.csv")
        i += 1
    df_rank.to_csv(out_rank, index=False)
    print(f"Cleaned rankings saved to: {out_rank}")
else:
    print("Starrank.csv not found; skipping raw rankings cleaning.")

# Merge rankings into hunt_sorted_by_flags 
rank_path = "Star_Rankings_combined_clean.csv"
assert os.path.exists(rank_path), f"{rank_path} not found."

cands = glob("**/hunt_sorted_by_flags*.csv", recursive=True)
assert cands, "hunt_sorted_by_flags*.csv not found. Run Cell 3 first."
hunt_path = max(cands, key=os.path.getmtime)

rank = pd.read_csv(rank_path, dtype=str)
hunt = pd.read_csv(hunt_path, dtype=str)

# Ensure TICID exists in rankings
if "TICID" not in rank.columns:
    assert "TIC ID" in rank.columns, f"'TIC ID' not found in {rank_path}."
    rank["TICID"] = rank["TIC ID"].astype(str)

# Normalize TICIDs
def normalize_ticid(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).replace("\u00A0"," ").strip()
    if s == "":
        return pd.NA
    m = re.fullmatch(r"(\d+)(?:\.0+)?", s)
    if m:
        return m.group(1)
    m = re.search(r"\d+", s)
    return m.group(0) if m else s

rank["TICID"] = rank["TICID"].map(normalize_ticid)
assert "TICID" in hunt.columns, f"'TICID' not found in {hunt_path}."
hunt["TICID"] = hunt["TICID"].map(normalize_ticid)

# De-duplicate rankings on TICID
rank = (rank.sort_values(list(rank.columns))
            .dropna(subset=["TICID"])
            .drop_duplicates(subset=["TICID"], keep="first"))

# Merge (left = keep all hunt rows)
merged = hunt.merge(
    rank.drop(columns=["TIC ID"], errors="ignore"),
    on="TICID",
    how="left",
    suffixes=("", "_rank"),
)

base = "hunt_merged_rankings.csv"
out = base
i = 1
while os.path.exists(out):
    stem, ext = os.path.splitext(base)
    out = f"{stem}_{i}{ext}"
    i += 1
merged.to_csv(out, index=False)

print(f"Saved merged hunt+rankings to: {out}")
print(f"hunt rows: {len(hunt)} | rankings rows (unique TICID): {len(rank)} | merged rows: {len(merged)}")
matched = merged["Star-Cluster"].notna().sum() if "Star-Cluster" in merged.columns \
          else merged.filter(regex="_rank$").notna().any(axis=1).sum()
print(f"Rows with a rankings match: {matched}")

# Quick TICID diagnostics 
tic = merged["TICID"].astype(str).str.strip()
n_missing_tic = (tic.eq("") | merged["TICID"].isna()).sum()
n_dupe_tic = tic.duplicated().sum()
print(f"TICID stats -> missing: {n_missing_tic} | duplicates: {n_dupe_tic} | unique: {tic.nunique()}")
